# Preprocessing pipeline quickstart

This notebook builds PACE commands for the complete preprocessing pipeline. Choose `SINGLE` or `BATCH`, then either generate the Slurm files for inspection or submit the jobs.

## Start with the shared PACE Python

Najafi lab members should first use the shared Suite2p 1.x Python. A personal Conda environment is not required when this path is accessible:

```bash
export TWO_P_PYTHON=/storage/project/r-fnajafi3-0/grubin6/shared_envs/2p_preprocessing_qc_suite2p_1x/bin/python
"$TWO_P_PYTHON" -c "from importlib.metadata import version; print(version('suite2p'))"
```

The editable `PYTHON_BIN` variable below already uses this path.

## Select the launch mode

- `SESSION_MODE`: `"SINGLE"` uses `SINGLE_SESSION`; `"BATCH"` writes `BATCH_SESSIONS` to `SESSIONS_FILE`.
- `ACTION`: `"submit"` submits Slurm jobs; `"generate"` only creates the job files.
- `RUN_COMMAND`: keep `False` to preview the command. Set it to `True` only after replacing the example paths.

## PACE storage and batch size

Run the jobs on PACE compute nodes. For multi-session runs, stage raw TIFFs to `/storage/scratch1/3/<username>/` when practical and use scratch for `OUTPUT_ROOT`. Cedar and project storage are durable shared filesystems, but simultaneous Suite2p jobs can saturate shared read bandwidth or trigger I/O throttling because each job repeatedly reads TIFF stacks and writes a large binary movie.

Scratch provides better temporary job I/O, but it is not backed up and may be purged. Copy validated final results back to durable storage.

Each session creates up to six Slurm jobs. Do not submit a large dataset all at once. Start with roughly 5-10 sessions per batch, use fewer for very large recordings, monitor with `squeue -u "$USER"`, and submit the next group after confirming the current batch is behaving normally. The standard `--sessions-file` interface does not limit the number of simultaneously active session chains.

In [ ]:
from pathlib import Path
import getpass
import shlex
import subprocess

# Choose: "SINGLE" or "BATCH"
SESSION_MODE = "SINGLE"

# Choose: "generate" or "submit" on PACE.
ACTION = "generate"

# Safety switch: preview the command until all paths are correct.
RUN_COMMAND = False

REPO_ROOT = Path("/path/to/2p_imaging")
OUTPUT_ROOT = Path(f"/storage/scratch1/3/{getpass.getuser()}/2p_processing_results")

PYTHON_BIN = Path(
    "/storage/project/r-fnajafi3-0/grubin6/shared_envs/"
    "2p_preprocessing_qc_suite2p_1x/bin/python"
)
SINGLE_SESSION = Path("/path/to/raw/session_1")
BATCH_SESSIONS = [
    Path("/path/to/raw/session_1"),
    Path("/path/to/raw/session_2"),
    Path("/path/to/raw/session_3"),
]
SESSIONS_FILE = Path("preprocessing_sessions.txt")

TARGET_STRUCTURE = "neuron"  # neuron, dendrite, or cerebellum_lax
ACCOUNT = "gts-fnajafi3"
QOS = "embers"
RUN_NAME = "example_preprocessing_run"

## Build the command

All sessions in batch mode share the settings above. Use separate runs when target structure, channel configuration, or selected stages differ.

In [ ]:
session_mode = SESSION_MODE.upper()

if session_mode not in {"SINGLE", "BATCH"}:
    raise ValueError('SESSION_MODE must be "SINGLE" or "BATCH"')
if ACTION not in {"generate", "submit"}:
    raise ValueError('ACTION must be "generate" or "submit"')
command = [
    str(PYTHON_BIN),
    "-m",
    "utils_2p.preprocessing_qc_pipeline",
    ACTION,
]

if session_mode == "SINGLE":
    command.extend(["--session", str(SINGLE_SESSION)])
else:
    session_text = "\n".join(str(path) for path in BATCH_SESSIONS) + "\n"
    SESSIONS_FILE.write_text(session_text, encoding="utf-8")
    command.extend(["--sessions-file", str(SESSIONS_FILE.resolve())])

command.extend(
    [
        "--output-root", str(OUTPUT_ROOT),
        "--target-structure", TARGET_STRUCTURE,
        "--suite2p-version", "1.x",
        "--python-bin", str(PYTHON_BIN),
        "--account", ACCOUNT,
        "--qos", QOS,
        "--run-name", RUN_NAME,
    ]
)

print(shlex.join(command))

For a functional-only single-channel recording, add these arguments before running:

```python
command.extend([
    "--nchannels", "1",
    "--functional-chan", "1",
    "--no-label",
])
```

Use `--target-structure dendrite` for dendritic recordings. Use `--stages dff,summary` to regenerate only downstream outputs when the required earlier outputs already exist.

## Run the command

This cell does nothing while `RUN_COMMAND` is `False`. Before enabling it, verify the repository, Python, raw-session, and output paths printed above.

In [ ]:
if RUN_COMMAND:
    if not REPO_ROOT.is_dir():
        raise FileNotFoundError(f"Repository does not exist: {REPO_ROOT}")
    if not PYTHON_BIN.is_file():
        raise FileNotFoundError(f"Python executable does not exist: {PYTHON_BIN}")
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Preview only. Set RUN_COMMAND = True after checking all paths.")

## Rebuild the environment only if needed

Use these instructions only if the shared Python is unavailable or different package versions are required. On PACE, make Conda available and create the Suite2p 1.x environment from the repository YAML:

```bash
module load anaconda3/2023.03

conda env create \
  --prefix ~/conda/envs/2p_preprocessing_qc_suite2p_1x \
  --file utils_2p/environment-preprocessing-qc-suite2p-1x.yml
```

Without a checkout, download the YAML first:

```bash
curl -L -o environment-preprocessing-qc-suite2p-1x.yml \
  https://raw.githubusercontent.com/najafi-laboratory/2p_imaging/main/utils_2p/environment-preprocessing-qc-suite2p-1x.yml

conda env create \
  --prefix ~/conda/envs/2p_preprocessing_qc_suite2p_1x \
  --file environment-preprocessing-qc-suite2p-1x.yml
```

After creating a personal environment, change `PYTHON_BIN` above to its `bin/python` path.